Cohort analysis tells us when to talk to customers. RFM (Recency, Frequency, Monetary) segmentation tells us who we are talking to and what we should say.

In retail and e-commerce analytics, RFM is the absolute gold standard for behavioral segmentation. It shifts marketing from "spray and pray" to surgical precision.

# "Who are our highest-value loyalists we need to protect, and who are the slipping customers we need to win back with a targeted discount?"

### Phase 1: Getting raw numbers for every customer

In [ ]:
# Step 1: The pull, importing the connection, connecting to the warehouse and pulling orders table.
import pandas as pd
from db_connections import connect

con = connect.get_warehouse_connection("../mean_mug_analytics/mean_mug.duckdb")
df_orders = con.execute("SELECT * FROM fct_orders").df()
df_orders.head()

Successfully connected to warehouse: ../mean_mug_analytics/mean_mug.duckdb


,order_id,customer_id,store_location_id,promo_id,order_date,order_time,payment_type_lower,total_amount,total_items_in_basket
0,1,1,1,<NA>,2024-01-26,09:21:00,app,7.50,2
1,2,1,2,2,2024-02-12,07:54:00,app,4.25,1
2,3,2,2,2,2024-02-08,09:22:00,app,16.00,4
3,4,2,1,<NA>,2024-03-26,17:06:00,card,4.25,1
4,5,3,2,<NA>,2024-03-12,13:19:00,cash,10.00,3


In [ ]:
# Step 2: Establish "today". To calculate Recency (how many days it has been since their last purchase), we need a reference date. 
# Today will be max order_date + 1 day, since datetime.now() will fail (latest year is 2024)
snapshot_date = df_orders['order_date'].max() + pd.DateOffset(days=1)
snapshot_date

Timestamp('2024-04-01 00:00:00')

In [ ]:
# Step 3: The Aggregation (Calculate R,F,and M)
RFM = df_orders.groupby('customer_id').agg(
    R= ('order_date', 'max'),
    F= ('order_id','nunique'),
    M= ('total_amount', 'sum')
)
# Overwrite the "R" column by subtracting, since cannot subtract in agg func
RFM['R'] = (snapshot_date - RFM['R']).dt.days

RFM

,R,F,M
customer_id,,,
1,49,2,11.75
2,6,2,20.25
3,7,3,51.00
4,8,2,8.50
5,2,2,32.50
...,...,...,...
995,10,1,19.50
996,1,2,10.75
997,1,3,20.75


### Phase 2: Scoring the customers (1-5) and assigning them into business buckets ("VIP","Churn Risk",etc.).

5 is the absolute best behavior. 1 is the worst behavior. To do this fairly, we don't guess the thresholds. We let the data divide itself into equal buckets of 20% (quintiles). The top 20% of spenders get a Monetary score of 5. The bottom 20% get a 1.

In [ ]:
RFM["R_Score"] = pd.qcut(RFM["R"],5,labels=[5,4,3,2,1])
# "Frequency" metrics where many customers have exactly 1 purchase, pd.qcut struggles to create equal-sized bins.
# Adding .rank(method="first") same as ROW_NUMBER() in SQL, so Pandas can rank each duplicate row by first seen order. 
RFM["F_Score"] = pd.qcut(RFM["F"].rank(method='first'),5,labels=[1,2,3,4,5])
RFM["M_Score"] = pd.qcut(RFM["M"],5,labels=[1,2,3,4,5])
RFM.head()


,R,F,M,R_Score,F_Score,M_Score
customer_id,,,,,,
1,49,2,11.75,1,2,1
2,6,2,20.25,3,2,2
3,7,3,51.00,3,3,4
4,8,2,8.50,3,2,1
5,2,2,32.50,5,2,3


In [ ]:
# Step 4: Concat the columns together:
RFM['RFM_Score'] = RFM['R_Score'].astype(str) + RFM['F_Score'].astype(str) + RFM['M_Score'].astype(str)

# Step 5: The Business Segments (To keep it actionable, marketers typically map segments using just Recency and Frequency (Monetary is often highly correlated with Frequency anyway)

def assign_segment(row):
    # 1. Grab R and F scores
    r = int(row['R_Score'])
    f = int(row['F_Score'])

    # 1. VIP
    if r == 5 and f == 5:
        return "VIP"

    # 2. Map the "Champions" (R= 4-5, F= 4-5)
    elif r >= 4 and f >= 4:
        return "Champion"

    # 3. Map the "Recent/New" (R=4-5,F=1-2)
    elif r >= 4 and f <= 2:
        return "Recent/New"

    # 4. "Loyal Customers": Buy often, but maybe haven't bought in a hot minute (R = 2-3, F = 4-5)
    elif (r == 2 or r==3) and f >= 4:
        return "Loyal Customers"

    # 5. At Risk": Used to buy often, haven't returned in a long time (R = 1-2, F = 3-5)
    elif r <= 2 and f >= 3:
        return "At Risk"

    # 6. "Hibernating/Lost": Bought a long time ago, and rarely bought anyway (R = 1-2, F = 1-2)
    elif r <= 2 and f <= 2:
        return "Lost"

    # 7. Everyting else in the middle
    else:
        return "Potential Loyalists"

# Create a new column by applying the func
RFM["Segment"] = RFM.apply(assign_segment, axis=1)



display(RFM[['R_Score', 'F_Score', 'Segment']].head(10))

,R_Score,F_Score,Segment
customer_id,,,
1,1,2,Lost
2,3,2,Potential Loyalists
3,3,3,Potential Loyalists
4,3,2,Potential Loyalists
5,5,2,Recent/New
6,2,1,Lost
7,5,2,Recent/New
9,5,4,Champion
10,2,2,Lost


In [ ]:
# Flatten DF
df = RFM.reset_index()
# Drop 'level_0' and 'index' if they exist, ignoring errors if they don't
df = df.drop(columns=['level_0', 'index'], errors='ignore')
df


,customer_id,R,F,M,R_Score,F_Score,M_Score,RFM_Score,Segment
0,1,49,2,11.75,1,2,1,121,Lost
1,2,6,2,20.25,3,2,2,322,Potential Loyalists
2,3,7,3,51.00,3,3,4,334,Potential Loyalists
3,4,8,2,8.50,3,2,1,321,Potential Loyalists
4,5,2,2,32.50,5,2,3,523,Recent/New
...,...,...,...,...,...,...,...,...,...
854,995,10,1,19.50,3,2,2,322,Potential Loyalists
855,996,1,2,10.75,5,3,1,531,Potential Loyalists
856,997,1,3,20.75,5,4,2,542,Champion
857,998,1,2,33.25,5,3,3,533,Potential Loyalists


In [ ]:
from db_connections import bigquery_export
bigquery_export.push_to_warehouse(df, "mart_rfm")

Uploading mart_rfm to BigQuery...


c:\Users\musta\OneDrive - University of Tennessee\mean_mug_project_july_2026\venv\Lib\site-packages\pandas_gbq\schema\pandas_to_bigquery.py:162: UserWarning: Could not determine the type of columns: R_Score, F_Score, M_Score
  warnings.warn(msg)


Success: mart_rfm is live!


In [ ]:
con.close()